In [1]:
from datasets import load_dataset

from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

''' The load_dataset is used for loading datasets in ML-reasy format.'''

''' The GPT2Tokenizer is used to convert the text into tokens. Hence, whatever prompts
are given to the model, they re converted into token ids'''

'''The GPT2LMHeadModel predicts the next tokens, given previous tokens.'''

'''The Trainer handles training engine. It includes, training loop, backpropagation, loss computation
checkpoint saving and evaluation.'''

'''TrainingArguments stores all the training configuration ehich includes learning rate, batch size, number of epochs etc etc.'''

'''DataCollatorForLanguageModeling prepares batches for casual language modelling'''

D:\Resume projects\mood-to-playlist\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'DataCollatorForLanguageModeling prepares batches for casual language modelling'

In [2]:
MODEL_NAME = "distilgpt2"

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

dataset = load_dataset(
    "json",
    data_files = "../../data/training/train.jsonl",
    split = "train"
)

In [3]:
def tokenize(batch):
    return tokenizer(batch["text"],
                     truncation = True, # Cuts off text longer than max_length
                     padding = "max_length", # Pads shorter sequences up to max_length
                     max_length = 256 # Sets fixed sequence length
                     )

In [4]:
tokenized = dataset.map(tokenize,
                        batched = True,
                        remove_columns = ["text"]
                        )

In [5]:
print(tokenized)

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 50
})


In [6]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer = tokenizer,
    mlm = False # mlm is Masked Language Modeling. GPT-2 uses Casual Language Modeling.
                # mlm is used by BERT
                # Casual Language Modeling means predict next tokens based on the previous tokens.
)

'''A data collator:
    Runs just before each training batch
    Takes tokenized samples
    Assembles them into model-ready batches'''

'A data collator:\n    Runs just before each training batch\n    Takes tokenized samples\n    Assembles them into model-ready batches'

In [7]:
training_args = TrainingArguments(
    output_dir = "../../models/gpt2_finetuned",
    overwrite_output_dir = True,
    num_train_epochs = 5,
    per_device_train_batch_size = 2,
    learning_rate = 5e-5,
    logging_steps = 10,
    save_steps = 200,
    save_total_limit = 2,
    report_to = "none"
)

In [8]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized,
    data_collator = data_collator
)

In [9]:
trainer.train()
trainer.save_model("../../models/gpt2_finetuned")

D:\Resume projects\mood-to-playlist\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.964600
20,2.784300
30,2.214500
40,2.026300
50,1.897100
60,1.661000
70,1.616600
80,1.514000
90,1.454200
100,1.419900
